In [2]:
import os

# see: https://docs.nvidia.com/deeplearning/frameworks/tensorflow-user-guide/index.html
os.environ['TF_GPU_ALLOCATOR']='cuda_malloc_async'
# and turning off debugging info to clean up output during final run
os.environ['TF_CPP_MIN_LOG_LEVEL']='3'

import tensorflow as tf
import onnxruntime as ort
import numpy as np
import tf2onnx
import onnx

# we also want to use mixed precision so
tf.keras.mixed_precision.set_global_policy(tf.keras.mixed_precision.Policy('mixed_float16'))
#from modules.datastore import load_datasets  # noqa: E402
#from modules.helper_functions import display_samples,eval_model,show_incorrect_predictions  # noqa: E402

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3090, compute capability 8.6


In [41]:
# see: https://onnxruntime.ai/docs/tutorials/tf-get-started.html

# load our saved model
model = tf.keras.models.load_model('results/models/export/to_export.keras')

input_signature = [tf.TensorSpec([None, 512, 512, 3], tf.float32, name='x')]

# Use from_function for tf functions
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature)
onnx.save(onnx_model, "results/models/export/exported.onnx")

In [49]:
# Change shapes and types to match model
input1 = np.zeros((1, 512, 512, 3), np.float32)

# Start from ORT 1.10, ORT requires explicitly setting the providers parameter if you want to use execution providers
# other than the default CPU provider (as opposed to the previous behavior of providers getting set/registered by default
# based on the build flags) when instantiating InferenceSession.
# Following code assumes NVIDIA GPU is available, you can specify other execution providers or don't include providers parameter
# to use default CPU provider.
sess = ort.InferenceSession("results/models/export/exported.onnx", providers=["CUDAExecutionProvider"])

# Set first argument of sess.run to None to use all model outputs in default order
# Input/output names are printed by the CLI and can be set with --rename-inputs and --rename-outputs
# If using the python API, names are determined from function arg names or TensorSpec names.
input_name = sess.get_inputs()[0].name
results_ort = sess.run(None, {input_name: input1})
results_tf = model(input1)

ort_prediction = results_ort[0][0]
tf_prediction = results_tf[0]
difference = abs(ort_prediction - tf_prediction)


print(f"ONNX prediction: {ort_prediction}")
print(f"Tensorflow prediction: {tf_prediction}")

assert(difference < 1e5)

print(f"Results match with difference of {difference}")

ONNX prediction: [0.35285008]
Tensorflow prediction: [0.35285005]
Results match with difference of [2.9802322e-08]
